## Entrenamiento de Autoencoders

Este script permite entrenar modelos de la clase Autoencoder sobre datos de la NBA, con el objetivo de generar embeddings de jugadores y analizarlos posteriormente.



Puede entrenar un autoencoder directamente sobre los datos originales o sobre embeddings generados por otro autoencoder de referencia (archivo_AE_ref), lo que resulta útil para tareas como reducción de dimensionalidad o análisis jerárquico de representaciones.



Funcionalidad principal:
- Carga un autoencoder y, opcionalmente, un autoencoder de referencia para obtener embeddings.
- Procesa los datos de entrenamiento y test, generando los tensores de entrada adecuados.
- Permite configurar hiperparámetros de entrenamiento y opciones de guardado.
- Realiza el entrenamiento del autoencoder y evalúa la mejora sobre el conjunto de test.
- Permite guardar el modelo actualizado y, opcionalmente, sus submodelos encoder y decoder.

---------------------------

### Preparación

Empezamos por importar las librerias, clases y funciones que vamos a usar.

In [ ]:
import torch
import torch.nn.functional as F
from MLP import MLP
from Autoencoder import Autoencoder 
from Guardar_Cargar import guardar_modelo, cargar_modelo
from Procesar_datos_AE import procesar_datos


### Selección de modelos

El primer paso es elegir el modelo que se va a entrenar, y si es necesario, el modelo de referencia.

In [21]:
archivo_autoencoder =r"redes_disponibles/pruebas/prueba.json"                    # Nombre del AE que se quiere entrenar

entrenar_sobre_AE_ref = False               # En caso de True: se entrenara el AE sobre los embeddings de referencia
archivo_AE_ref = r""                        # Nombre del AE de referencia (si se quiere entrenar sobre embeddings)

#### Opciones guardado

Ahora seleccionamos si guradar el modelo después de entrenarlo. También seleccionamos si guardar el modelo a pesar de no mejorar el rendimiento sobre el test.

In [22]:
save_after_training = True             # En caso de True: se guarda cuando mejora el error respecto 
override_guardado = False              # En caso de True: se guarda aunque no mejore el error (si el anterior es True)


En caso de querer guardar o sobreescribir los submodelos del AE lo seleccionamos a continuación.

In [23]:
sobreescribir_submodelos = False       # En caso de True: Se sobreescriben archivos de encoder y decoder.
archivo_encod = r"" 
archivo_decod = r"" 


### Selección de datos entrenamiento

En esta sección seleccionamos y procesamos los datos con los que entrenar el modelo y los datos de test para validación.

El primer paso es definir el tipo de estadísticas de la nba que va a analizar el modelo. Las hemos dividido en tres tipos:
- Estadísticas completas: incluye todos los tipos de estadísticas, nos da una imagen más general pero ante la complejidad puede ser dificil de interpretar.
- Estadísticas de volumen: incluye las estadísticas tradicionales pero unicamente en volumen. Util para analizar rendimiento global directo pero se pierde información de eficiencias.
- Estadísticas de tiro: Incluye las estadísticas tradicionales junto a porcentajes de tiro y distribución de los mismos. Util para analizar el tipo de jugador ofensivo, pero esta faceta puede opacar otras en un análisis más general.

Para seleccionar el tipo de estadística usa: 'completo', 'volumen' o 'tiro'.

In [24]:
tipo_estadisticas = "tiro" # Tipo de estadísticas a utilizar

A continuación se eligén los datasets a usar para entrenamiento y validación. Se encuentran varios disponibles en la carpeta datasets\nba.

In [25]:
archivo_entrenamiento = r"datasets\nba\finales\nba19_24_per100_entrenamiento.csv"
archivo_test = r"datasets\nba\finales\test_per100_nba18_19.csv" 


Una vez seleccionados los datasets procesamos los datos para el entrenamiento y test.

In [26]:
xs_orig, etiquetas_emb = procesar_datos(archivo_entrenamiento,tipo_estadisticas)
xs_test_orig, etiquetas_test = procesar_datos(archivo_test,tipo_estadisticas)


✅ No se encontraron NaNs en X.Total de jugadores válidos: 2112

 Datos de datasets\nba\finales\nba19_24_per100_entrenamiento.csv procesados correctamente (tipo_stats=tiro, n_columnas=30). 

✅ No se encontraron NaNs en X.Total de jugadores válidos: 415

 Datos de datasets\nba\finales\test_per100_nba18_19.csv procesados correctamente (tipo_stats=tiro, n_columnas=30). 



### Seleccion de hiperparametros

En esta sección detallaremos los hipérparametros para el entrenamiento del modelo.
Comenzamos con el número de pasos de entrenamiento y el tamaño del paso.

In [32]:
stp_n = 1000          # Número de pasos de entrenamiento
stp_sz = 0.001         # Tamaño del paso (learning rate)
batch_sz = 64          # Tamaño del batch (por defecto si es None, todo el dataset)


A continuación elegimos la función de perdida que optimizar durante el entrenamiento así como hiperparámetros de penalización en nuestro modelo. Si no se desea emplear estas penalizaciones basta con dejarlas en 0.

In [28]:
loss_f = F.mse_loss    # Función de pérdida

# Regularización sobre representación latente(sparsity)
beta_l1 = 0.0          # Peso de la penalización L1 sobre la representación latente.
beta_kl = 0.0          #  Sparsity por KL divergence
# Regularización sobre pesos
lambda_l2 = 1e-4       # Regularización L2 de pesos


### Entrenamiento y test

El primer paso antes de entrenar es cargar el modelo.

In [29]:
print(f"\nSe va entrenar el modelo '{archivo_autoencoder}'.")
NN = cargar_modelo(archivo_autoencoder)


Se va entrenar el modelo 'redes_disponibles/pruebas/prueba.json'.


#### Procesamiento datos finales

En caso de querer entrenar el modelo usando otro como referencia lo importamos y pasamos los datos de entrenamiento por su encoder. En caso contrario mantenemos los datos de entrenamiento originales.

In [30]:
if entrenar_sobre_AE_ref:
        ae_emb = cargar_modelo(archivo_AE_ref)
        encod_emb = ae_emb.encoder

        xs_train = encod_emb(xs_orig).detach()
        xs_test  =  encod_emb(xs_test_orig).detach()
        ys_test  = xs_test
        print(f"Se han procesado los datos con el modelo de referencia {archivo_AE_ref}.")
    
else:
        xs_train = xs_orig
        xs_test  = xs_test_orig
        ys_test  = xs_test
        print("Se han mantenido los datos originales.")

Se han mantenido los datos originales.


#### Análisis pérdida inicial

Antes de entrenar el modelo evaluamos su error inicial sobre el test acorde a la función de pérdida seleccionada para observar su mejora.

In [33]:
with torch.no_grad():
        pred_test_init = NN(xs_test)
        loss_init = loss_f(pred_test_init, ys_test) 
print(f"\nEl modelo '{archivo_autoencoder}' tiene una perdida inicial sobre el test: {loss_init}.")



El modelo 'redes_disponibles/pruebas/prueba.json' tiene una perdida inicial sobre el test: 7.603579998016357.


#### Entrenamiento

Procedemos a entrenar el modelo. Esto puede llevar un tiempo según los hiperparámetros elegidos.

In [35]:
print(f"\nIniciamos entrenamiento de {stp_n} pasos de el modelo '{archivo_autoencoder}'.\n") 

NN.train_model(xs_train,stp_n,stp_sz,loss_f,batch_sz,beta_l1,beta_kl,lambda_l2)

print(f"\nFinalizado entrenamiento de {stp_n} pasos de el modelo '{archivo_autoencoder}'.\n") 



Iniciamos entrenamiento de 1000 pasos de el modelo 'redes_disponibles/pruebas/prueba.json'.

Paso 0 | Loss total: 0.035370 (Recon: 0.034983, L1: 0.000000, KL: 0.000000, L2: 3.869048)
Paso 50 | Loss total: 0.035205 (Recon: 0.034818, L1: 0.000000, KL: 0.000000, L2: 3.869300)
Paso 100 | Loss total: 0.035074 (Recon: 0.034687, L1: 0.000000, KL: 0.000000, L2: 3.869546)
Paso 150 | Loss total: 0.034943 (Recon: 0.034556, L1: 0.000000, KL: 0.000000, L2: 3.869783)
Paso 200 | Loss total: 0.034855 (Recon: 0.034468, L1: 0.000000, KL: 0.000000, L2: 3.870009)
Paso 250 | Loss total: 0.034758 (Recon: 0.034371, L1: 0.000000, KL: 0.000000, L2: 3.870230)
Paso 300 | Loss total: 0.034666 (Recon: 0.034279, L1: 0.000000, KL: 0.000000, L2: 3.870444)
Paso 350 | Loss total: 0.034578 (Recon: 0.034191, L1: 0.000000, KL: 0.000000, L2: 3.870655)
Paso 400 | Loss total: 0.034533 (Recon: 0.034146, L1: 0.000000, KL: 0.000000, L2: 3.870859)
Paso 450 | Loss total: 0.034455 (Recon: 0.034068, L1: 0.000000, KL: 0.000000, L2:

#### Evaluacíon pérdida final

Una vez entrenado el modelo evaluamos su error final sobre el test para ver si ha mejorado el rendimiento.

In [36]:
with torch.no_grad():
        pred_test_fin = NN(xs_test)
        loss_final = loss_f(pred_test_fin, ys_test) 
print(f"\nEl modelo '{archivo_autoencoder}' tiene una perdida final sobre el test: {loss_final}.\n")


El modelo 'redes_disponibles/pruebas/prueba.json' tiene una perdida final sobre el test: 0.03337843716144562.



### Guardado

Antes de guardar tenemos la opción de modificar la descripción del modelo y/o añadir información sobre el entrenamiento. 

In [37]:
descripcion = f" Entrenamiento de {stp_n} pasos de tamano {stp_sz} con funcion de perdida {loss_f.__name__} en batches de {batch_sz} y valores beta_l1={beta_l1},beta_kl={beta_kl}, lambda_l2={lambda_l2}."
añadir_descripcion = True          # Añade a la descripcion ya existente
sustituir_desc = False             # CUIDADO, SI TRUE ELIMINA LA DESCRIPCIÓN YA EXISTENTE
añadir_info_mejora = True          # Añade informacion de como ha mejorado/empeorado el modelo sobre el test dado


In [38]:

if añadir_descripcion:
        if añadir_info_mejora:
            descripcion += f"\n El modelo ha pasado de {loss_init} a {loss_final} sobre el test {archivo_test} tras entrenar con el dataset {archivo_entrenamiento}."
        if sustituir_desc:
            NN.description = descripcion
        else:
            NN.add_descript(descripcion)


Finalmente guardamos el modelo según los ajustes seleccionados.

In [39]:
if save_after_training:
    
    if loss_final < loss_init or override_guardado:
        print( f"El error del modelo '{archivo_autoencoder}' sobre el test ha mejorado o se ha decidido sobreescribir y por tanto lo actualizamos.\n")
        guardar_modelo(NN,archivo_autoencoder)
            
        if sobreescribir_submodelos:
                guardar_modelo(NN.encoder,archivo_encod)
                guardar_modelo(NN.decoder,archivo_decod)
    else:
        print( f"El error del modelo '{archivo_autoencoder}' sobre el test no ha mejorarado y por tanto NO la actualizamos.\n")
else:
        print(f"Se ha decidido NO guardar la actualizacion del modelo '{archivo_autoencoder}.'\n")


El error del modelo 'redes_disponibles/pruebas/prueba.json' sobre el test ha mejorado o se ha decidido sobreescribir y por tanto lo actualizamos.

Autoencoder guardado en 'redes_disponibles/pruebas/prueba.json'

